การเรียนรู้ของเครื่องสำหรับอุตสาหกรรม -- ดร.วโรดม ตู้จินดา

# 1 บทนำ

ในงานอุตสาหกรรมทั่วไป วิศวกรสามารถออกแบบ/ใช้งานตัวควบคุมเช่น PID (Proportional Integrad Derivative) หรือ LQR (Linear Quadratic Regulator) ให้เครื่องจักรกลทำงานได้ตามวัตถุประสงค์ มีค่าเผื่อเสถียรภาพรวมถึงข้อกำหนดสมรรถนะตามที่กำหนด ในกรณีที่มีการรบกวนเขาสามารถใช้วิธีการเช่นตัวกรองคาลมานเข้าช่วย วิธีการเหล่านี้มีพื้นฐานอยู่บนโมเดลทางคณิตศาสตร์ของเครื่องจักร ที่อาจหาจากหลักการทางฟิสิกส์หรือวิธีการหาเอกลักษณ์ระบบ (system identification) แม้ไม่ถูกต้องสมบูรณ์ 100% แต่เป็นตัวแทนที่ดีของเครื่องจักรจริงได้ในระดับหนึ่ง

เนื้อหาของหนังสือเล่มนี้จะเกี่ยวข้องกับปัญหาที่แตกต่างออกไป คือในกรณีที่เราทราบฟิสิกส์เพียงบางส่วน หรือมีค่าใช้จ่ายสูงเกินไปในการหาโมเดล หรือต้องการทำนายผลจากสัญญาณที่ไม่สามารถอ่านได้โดยสายตาสำหรับงานอุตสาหกรรมประเภทต่างๆ ตัวอย่างเช่นในการบำรุงรักษาเชิงป้องกัน (preventive maintenance) ต้องการทราบว่าตลับลูกปืน (bearings) ของมอเตอร์เริ่มจะเสื่อมสภาพหรือยัง ในงานการผลิตอัตโนมัติ ต้องการตรวจสอบคุณภาพรอยเชื่อมโดยภาพจากกล้อง หรือความสามารถของมือจับ (gripper) ของแขนกลในการจับชิ้นงานที่มีรูปทรงทางเรขาคณิตแตกต่างกัน การเรียนรู้ของเครื่อง (machine learning เพื่อความกระชับจะใช้ตัวย่อ ML) สามารถตอบคำถามเหล่านี้ได้โดยการฟิตฟังก์ชันกับตัวอย่างแทนการอนุพัทธ์จากหลักการทางฟิสิกส์ ทั้งนี้ ML ไม่ใช่วิธีการที่จะมาแทนที่การควบคุมและประมาณค่า​ (control and estimation) แต่สามารถใช้ควบคู่กันเพื่อสร้างระบบที่ดีที่สุด

ในบทแรกนี้จะเป็นการนิยามคำศัพท์ กระแสงาน (workflow) เครื่องมือที่ใช้ในหนังสือ และสร้างเครื่องขนาดเล็กที่สามารถเรียนรู้ได้จากเริ่มต้น คือ ประสาทเทียม (artificial neuron) จากนั้นสาธิตโมเดลที่เหมือนทำงานถูกต้องแต่ด้วยเหตุผลที่ผิด คือการตรวจจับความผิดพลาดที่มีความแม่นยำ 99% แต่เป็นเพียงวัดว่าสายพานมีภาระมากเพียงใดเท่านั้น

___
## วัตถุประสงค์ในการเรียนรู้

เป้าหมายในการศึกษาของบทนี้คือช่วยให้ผู้อ่านสามารถ

1. อธิบายได้ว่า ML ช่วยเสริมการควบคุมและประมาณค่าบนพื้นฐานโมเดลอย่างไร และไม่เหมาะสมในการใช้ในงานประเภทใด
2. นิยามการเรียนรู้ในรูปของ *งาน (task) ตัววัดสมรรถนะ (performance measure)* และ *ประสบการณ์ (experience)* และสามารถแยกประเภทปัญหาการเรียนรู้ในงานอุตสาหกรรม
3. เข้าใจสัญกรณ์ที่ใช้ในหนังสือ: *เมทริกซ์ลักษณะเด่น (feature matrix)* $X$, *เวกเตอร์เป้าหมาย (target vector)* $y$ *โมเดล (model) ค่าสูญเสีย (loss) ชุดฝึกและทดสอบ (training and test sets)*
4. อิมพลิเมนต์ *เพอร์เซปตรอน (perceptron)* และ *แอดาไลน์ (adaline)*
5. ฝึกและประเมิน *ตัวจำแนก (classifier)* โดยใช้แบบ `fit/predict` ภายใน `Pipeline` บนเครื่องมือ `scikit-learn`
6. ออกแบบการทดสอบเพื่อแยกโมเดลที่เรียนรู้ฟิสิกส์จากโมเดลที่เพียงเรียนรู้ความบังเอิญในข้อมูล
___

## 1.1 ตัวอย่างปัญหาเชิงวิศวกรรม

> **หมายเหตุ :** หนังสือนี้จัดทำในรูปของโน๊ตบุกเนื่องจากต้องการให้ผู้อ่านสามารถรันโค้ดไปพร้อมอ่านเนื้อหา โดยสามารถแก้ไขโค้ดเพื่อดูการเปลี่ยนแปลงของเอาต์พุตได้ในทันที ผู้เขียนเชื่อว่าการเรียนรู้แบบเชิงโต้ตอบนี้จะช่วยกระตุ้นความสนใจได้ดีกว่าการอ่านเพียงอย่างเดียว

โรงงานผลิตแพ็กเกจมีสายพานลำเลียงที่ขับเคลื่อนโดยมอเตอร์เหนี่ยวนำผ่านชุดเกียร์ ที่ทางด้านตลับลูกปืนของมอเตอร์มี *มาตรความเร่ง
(accelerometer)* ติดตั้งอยู่ และมีอุปกรณ์เอดจ์คอมพิวเตอร์ (edge computer) สำหรับบันทึกหน้าต่างการสั่นสะเทือนเป็นเวลา 0.2 วินาที (หรืออัตรา 5 KHz) เป็นจำนวนหลายครั้งในแต่ละกะการทำงาน เมื่อวงแหวนนอก (outer race) ของตลับลูกปืนเกิดเป็นหลุม (pit)หรือมีการหลุดร่อนของโลหะผิวหน้า (spall) มอเตอร์ยังสามารถจะหมุนได้อยู่หลายสัปดาห์ จนกว่าจะใช้งานต่อไปไม่ได้และต้องหยุดสายการผลิตทั้งหมด  

หน่วยซ่อมบำรุงต้องการการแจ้งเตือนที่ระบุว่าแต่ละหน้าต่างข้อมูลนั้นยังปกติดี หรือว่ามีข้อบกพร่องกับวงแหวนนอกแล้ว เพื่อที่จะจัดตารางการบำรุงรักษาตลับลูกปืนได้อย่างเหมาะสมในการหยุดงานตามแผนในครั้งหน้า (แทนที่จะต้องหยุดสายการผลิตเพื่อซ่อมตลับลูกปืนซึ่งจะใช้เวลานานและมีค่าใช้จ่ายสูงกว่า)

ในตัวอย่างนี้จะใช้ข้อมูลจากการจำลองโดยหลักการฟิสิกส์ของการสั่นสะเทือน ซึ่งสร้างจากโค้ดในโน๊ตบุกนี้ทั้งหมดทำให้ไม่ต้องดาวน์โหลดข้อมูลจากที่อื่น เริ่มจากการนำเข้าแพ็กเกจที่ต้องการใช้งาน (เลือกใช้สภาวะแวดล้อมที่ติดตั้งจากภาคผนวก A)









In [1]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
import sklearn
from scipy.stats import kurtosis

print("python      ", sys.version.split()[0])
print("numpy       ", np.__version__)
print("pandas      ", pd.__version__)
print("scipy       ", scipy.__version__)
print("matplotlib  ", matplotlib.__version__)
print("scikit-learn", sklearn.__version__)
try:
    import torch
    print("torch       ", torch.__version__, "(needed from Chapter 8 on)")
except ImportError:
    print("torch        not installed (not needed until Chapter 8)")

# plot style shared by every chapter of the book
plt.rcParams.update({
    "figure.figsize": (9, 3.6), "figure.dpi": 110,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": "#e1e0d9",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7", "font.size": 10, "legend.frameon": False,
})
BLUE, ORANGE, AQUA, YELLOW, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#898781"
CLASS_COLORS = [BLUE, ORANGE]          # healthy, faulty
CLASS_NAMES = ["healthy", "outer-race defect"]

os.makedirs("figures", exist_ok=True)


def savefig(name):
    """Save the current figure for the chapter text (figures/<name>.png)."""
    plt.savefig(f"figures/{name}.png", dpi=160, bbox_inches="tight")

python       3.13.5
numpy        2.3.1
pandas       2.3.1
scipy        1.16.0
matplotlib   3.10.5
scikit-learn 1.7.0
torch        2.12.0 (needed from Chapter 8 on)


<div align="center">
<img src="https://raw.githubusercontent.com/dewdotninja/sharing-github/refs/heads/master/dewninja_logo50.jpg" alt="dewninja"/>
</div>
<div align="center">dew.ninja 2027</div>